# Recommender Systems / Neural Networks

In [2]:
import numpy as np
from numpy.typing import NDArray
from typing import Tuple
import numpy.ma as ma
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
import torchvision as tv
import pandas as pd
import pickle
from numpy import genfromtxt
from collections import defaultdict
import csv
import tabulate
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.precision", 1)

Matplotlib is building the font cache; this may take a moment.


In [3]:
data_path = './data/recommender_system/'

def load_data():
    item_train = genfromtxt(f'{data_path}/content_item_train.csv', delimiter=',')
    user_train = genfromtxt(f'{data_path}/content_user_train.csv', delimiter=',')
    y_train    = genfromtxt(f'{data_path}/content_y_train.csv', delimiter=',')
    with open(f'{data_path}/content_item_train_header.txt', newline='') as f:    #csv reader handles quoted strings better
        item_features = list(csv.reader(f))[0]
    with open(f'{data_path}/content_user_train_header.txt', newline='') as f:
        user_features = list(csv.reader(f))[0]
    item_vecs = genfromtxt(f'{data_path}/content_item_vecs.csv', delimiter=',')
       
    movie_dict = defaultdict(dict)
    count = 0
#    with open('{data_path}/movies.csv', newline='') as csvfile:
    with open(f'{data_path}/content_movie_list.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: 
                count +=1  #skip header
                #print(line) 
            else:
                count +=1
                movie_id = int(line[0])  
                movie_dict[movie_id]["title"] = line[1]  
                movie_dict[movie_id]["genres"] =line[2]  

    with open(f'{data_path}/content_user_to_genre.pickle', 'rb') as f:
        user_to_genre = pickle.load(f)

    return(item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre)

def split_str(ifeatures, smax):
    ofeatures = []
    for s in ifeatures:
        if ' ' not in s:  # skip string that already have a space            
            if len(s) > smax:
                mid = int(len(s)/2)
                s = s[:mid] + " " + s[mid:]
        ofeatures.append(s)
    return(ofeatures)


In [4]:
# Load Data, set configuration variables
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
scaledata = True  # applies the standard scalar to data if true
print(f"Number of training vectors: {len(item_train)}")
print(f'Shape: item_train: {item_train.shape}, user_train: {user_train.shape}, y_train: {y_train.shape}')

Number of training vectors: 58187
Shape: item_train: (58187, 17), user_train: (58187, 17), y_train: (58187,)


/var/folders/rp/5h6kshv97z19k30t9r4499gw0000gn/T/ipykernel_19326/86960790.py:29: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  user_to_genre = pickle.load(f)


In [6]:
def pprint_train(x_train, features,  vs, u_s, maxcount = 5, user=True):
    """ Prints user_train or item_train nicely """
    if user:
        flist = [".0f",".0f",".1f", 
                 ".1f", ".1f", ".1f", ".1f",".1f",".1f", ".1f",".1f",".1f", ".1f",".1f",".1f",".1f",".1f"]
    else:
        flist = [".0f",".0f",".1f", 
                 ".0f",".0f",".0f", ".0f",".0f",".0f", ".0f",".0f",".0f", ".0f",".0f",".0f",".0f",".0f"]

    head = features[:vs]
    if vs < u_s: print("error, vector start {vs} should be greater then user start {u_s}")
    for i in range(u_s):
        head[i] = "[" + head[i] + "]"
    genres = features[vs:]
    hdr = head + genres
    disp = [split_str(hdr, 5)]
    count = 0
    for i in range(0,x_train.shape[0]):
        if count == maxcount: break
        count += 1
        disp.append( [ 
                      x_train[i,0].astype(int),  
                      x_train[i,1].astype(int),   
                      x_train[i,2].astype(float), 
                      *x_train[i,3:].astype(float)
                    ])
    table = tabulate.tabulate(disp, tablefmt='html',headers="firstrow", floatfmt=flist, numalign='center')
    return(table)

pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9


In [7]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8798,2004,3.8,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [8]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  4.  4.  3.5 3.5]


In [9]:
# scale training data

def standard_scaler(x: NDArray) -> NDArray:
    mean = x.mean(axis = 0)
    std = x.std(axis=0)
    std_safe = np.where(std == 0, 1, std)
    return (x - mean) / std_safe

if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    item_train = standard_scaler(item_train)
    user_train = standard_scaler(user_train)

In [10]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

movie/item training data shape: (46549, 17)
movie/item test  data shape: (11638, 17)


In [11]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
1,0,0.6,0.7,0.6,0.6,0.7,0.7,0.5,0.7,0.2,0.3,0.3,0.5,0.5,0.8,0.5
0,0,1.6,1.5,1.7,0.9,1.0,1.4,0.8,-1.2,1.2,1.2,1.6,0.9,1.4,1.2,1.0
0,0,0.8,0.6,0.7,0.5,0.6,0.6,0.3,-1.2,0.7,0.8,0.9,0.6,0.2,0.6,0.6
1,0,-0.1,0.2,-0.1,0.3,0.7,0.3,0.2,1.0,-0.5,-0.7,-2.1,0.5,0.7,0.3,0.0
-1,0,-1.3,-0.8,-0.8,0.1,-0.1,-1.1,-0.9,-1.2,-1.5,-0.6,-0.5,-0.6,-0.9,-0.4,-0.9


In [12]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train1 = scaler.transform(y_train.reshape(-1, 1))
ynorm_test1 = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train1.shape, ynorm_test1.shape)

def min_max_scaler(x: NDArray, min: float, max: float) -> NDArray:
    x_min = np.min(x)
    x_max = np.max(x)
    return (x - x_min) / (x_max - x_min) * (max - min) + min

ynorm_train = min_max_scaler(y_train, -1, 1).reshape(-1, 1)
ynorm_test = min_max_scaler(y_test, -1, 1).reshape(-1, 1)
print(ynorm_train.shape, ynorm_test.shape)

print(np.allclose(ynorm_train1, ynorm_train))
print(np.allclose(ynorm_test1, ynorm_test))

(46549, 1) (11638, 1)
(46549, 1) (11638, 1)
True
True


In [13]:
class EarlyStopping:
    def __init__(self, patience, min_delta):
        self.patience = patience
        self.min_delta = min_delta
        
        # 内部状态
        self.counter = 0
        self.best_cost = np.inf

    def step(self, cost: Tensor) -> bool:
        delta = (self.best_cost - cost).item()
        self.best_cost = cost
        if delta < self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                return True
            else:
                return False

        self.counter = 0

        return False


In [14]:
class RecommendationModel(nn.Module):
    def __init__(self, input_dim_user, input_dim_item, output_dim):
        super().__init__()  # 必须写
        self.user_nn = nn.Sequential(
            nn.Linear(input_dim_user, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

        self.item_nn = nn.Sequential(
            nn.Linear(input_dim_item, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, user_input, item_input) -> Tensor:
        vu = self.user_nn(user_input)
        vm = self.item_nn(item_input)
        vu = F.normalize(vu, p=2, dim=1)
        vm = F.normalize(vm, p=2, dim=1)
        
        dot_product = torch.sum(vu * vm, dim=1, keepdim=True)
        
        return dot_product

print(user_train.shape, item_train.shape)
u_n = user_train.shape[1] - u_s
i_n = item_train.shape[1] - i_s
model = RecommendationModel(input_dim_user=u_n, input_dim_item=i_n, output_dim=32)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 1000
print_point = 20
early_stopping = EarlyStopping(patience=5, min_delta=1e-5)

user_train_tensor = torch.tensor(user_train[:,u_s:], dtype=torch.float32)
item_train_tensor = torch.tensor(item_train[:,i_s:], dtype=torch.float32)
ynorm_train_tensor = torch.tensor(ynorm_train, dtype=torch.float32)
model.train()
for epoch in range(epochs):
    pred = model(user_train_tensor, item_train_tensor)
    loss = criterion(pred, ynorm_train_tensor)
    if early_stopping.step(loss) is True:
        print(f'Early Stopping at Epoch {epoch+1}')
        break
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % print_point == 0:
        print(f"Epoch [{epoch+1:6d}/{epochs:6d}] Loss: {loss.item():.8f}")

print(f"Done with final Loss: {loss.item():.8f}")

(46549, 17) (46549, 17)
Epoch [    20/  1000] Loss: 0.12793022
Epoch [    40/  1000] Loss: 0.11955599
Epoch [    60/  1000] Loss: 0.11679388
Epoch [    80/  1000] Loss: 0.11466079
Epoch [   100/  1000] Loss: 0.11197938
Epoch [   120/  1000] Loss: 0.10940874
Epoch [   140/  1000] Loss: 0.10727070
Epoch [   160/  1000] Loss: 0.10555238
Epoch [   180/  1000] Loss: 0.10408907
Epoch [   200/  1000] Loss: 0.10278216
Epoch [   220/  1000] Loss: 0.10156289
Epoch [   240/  1000] Loss: 0.10041089
Epoch [   260/  1000] Loss: 0.09932683
Epoch [   280/  1000] Loss: 0.09833375
Epoch [   300/  1000] Loss: 0.09737612
Epoch [   320/  1000] Loss: 0.09650713
Epoch [   340/  1000] Loss: 0.09582725
Epoch [   360/  1000] Loss: 0.09533720
Epoch [   380/  1000] Loss: 0.09437425
Epoch [   400/  1000] Loss: 0.09367028
Early Stopping at Epoch 405
Done with final Loss: 0.09412155


In [15]:
from torchinfo import summary
summary(model, input_size=[(1, u_n), (1, i_n)]) 

Layer (type:depth-idx)                   Output Shape              Param #
RecommendationModel                      [1, 1]                    --
├─Sequential: 1-1                        [1, 32]                   --
│    └─Linear: 2-1                       [1, 256]                  3,840
│    └─ReLU: 2-2                         [1, 256]                  --
│    └─Linear: 2-3                       [1, 128]                  32,896
│    └─ReLU: 2-4                         [1, 128]                  --
│    └─Linear: 2-5                       [1, 32]                   4,128
├─Sequential: 1-2                        [1, 32]                   --
│    └─Linear: 2-6                       [1, 256]                  4,352
│    └─ReLU: 2-7                         [1, 256]                  --
│    └─Linear: 2-8                       [1, 128]                  32,896
│    └─ReLU: 2-9                         [1, 128]                  --
│    └─Linear: 2-10                      [1, 32]                   4

In [19]:
def gen_user_vecs(user_vec, num_items):
    """ given a user vector return:
        user predict maxtrix to match the size of item_vecs """
    user_vecs = np.tile(user_vec, (num_items, 1))
    return(user_vecs)
def print_pred_movies(y_p, user, item, movie_dict, maxcount=10):
    """ print results of prediction of a new user. inputs are expected to be in
        sorted order, unscaled. """
    count = 0
    movies_listed = defaultdict(int)
    disp = [["y_p", "movie id", "rating ave", "title", "genres"]]

    for i in range(0, y_p.shape[0]):
        if count == maxcount:
            break
        count += 1
        movie_id = item[i, 0].astype(int)
        if movie_id in movies_listed:
            continue
        movies_listed[movie_id] = 1
        disp.append([y_p[i, 0], item[i, 0].astype(int), item[i, 2].astype(float),
                    movie_dict[movie_id]['title'], movie_dict[movie_id]['genres']])

    table = tabulate.tabulate(disp, tablefmt='html',headers="firstrow")
    return(table)

new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vecs = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

# user_vec = torch.tensor(gen_user_vecs(user_vec, len(item_vecs)), dtype=torch.float32)

def predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, ScalerUser, ScalerItem, scaledata=False):
    if scaledata:
        scaled_user_vecs = torch.tensor(standard_scaler(user_vecs[:,u_s:]), dtype=torch.float32)
        scaled_item_vecs = torch.tensor(standard_scaler(item_vecs[:,i_s:]), dtype=torch.float32)
        y_p = model(scaled_user_vecs, scaled_item_vecs)
    else:
        # 重要：把单个用户复制成和电影一样多的数量
        user_repeat = np.tile(user_vecs[:, u_s:], (item_vecs.shape[0], 1))
        user_tensor = torch.tensor(user_repeat, dtype=torch.float32)
        item_tensor = torch.tensor(item_vecs[:, i_s:], dtype=torch.float32)
        
        y_p = model(user_tensor, item_tensor)
        
    y_pu = scaler.inverse_transform(y_p.detach().numpy())

    if np.any(y_pu < 0) : 
        print("Error, expected all positive predictions")
        
    sorted_index = np.argsort(-y_pu, axis=0).reshape(-1).tolist()
    sorted_ypu   = y_pu[sorted_index]
    sorted_items = item_vecs[sorted_index]
    
    # 🔥 修复在这里！
    sorted_user  = np.tile(user_vecs, (len(sorted_index), 1))

    return (sorted_index, sorted_ypu, sorted_items, sorted_user)

model.eval()
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, item_train, user_train, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

y_p,movie id,rating ave,title,genres
4.13843,166643,3.8,Hidden Figures (2016),Drama
4.12784,187593,3.875,Deadpool 2 (2018),Action|Comedy|Sci-Fi
4.12623,150548,3.85,Sherlock: The Abominable Bride (2016),Action|Crime|Drama|Mystery|Thriller
4.12065,137857,3.63636,The Jungle Book (2016),Adventure|Drama|Fantasy
4.11652,134853,3.81395,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy
4.11377,117176,3.70588,The Theory of Everything (2014),Drama|Romance
4.11349,112556,3.71622,Gone Girl (2014),Drama|Thriller
4.10469,109374,3.77885,"Grand Budapest Hotel, The (2014)",Comedy|Drama
4.10345,115713,3.91071,Ex Machina (2015),Drama|Sci-Fi|Thriller


In [25]:
def sq_dist(a,b):
    d = sum(np.square(a-b))
    return (d)

def get_item_genre(item, ivs, item_features):
    offset = np.where(item[ivs:] == 1)[0][0]
    genre = item_features[ivs + offset]
    return(genre, offset)


a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

squared distance between a1 and b1: 0.0
squared distance between a2 and b2: 0.030000000000000054
squared distance between a3 and b3: 2


In [26]:
u_t = torch.tensor(user_train[:,u_s:], dtype=torch.float32)
i_t = torch.tensor(item_train[:,i_s:], dtype=torch.float32)
vms = model(u_t, i_t).detach().numpy()
print(f"size of all predicted movie feature vectors: {vms.shape}")

count = 50
vms = vms[:1000,:]
dim = len(vms)
dist = np.zeros((dim,dim))
for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

size of all predicted movie feature vectors: (46549, 1)


movie1,genres,movie2,genres
Save the Last Dance (2001),Drama,Anger Management (2003),Comedy
Save the Last Dance (2001),Romance,Catch Me If You Can (2002),Crime
"Wedding Planner, The (2001)",Comedy,Johnny English (2003),Comedy
"Wedding Planner, The (2001)",Romance,From Hell (2001),Horror
Hannibal (2001),Horror,"Punisher, The (2004)",Crime
Hannibal (2001),Thriller,Scooby-Doo (2002),Mystery
Saving Silverman (Evil Woman) (2001),Comedy,Showtime (2002),Comedy
Saving Silverman (Evil Woman) (2001),Romance,Adaptation (2002),Romance
Down to Earth (2001),Comedy,Shaolin Soccer (Siu lam juk kau) (2001),Action
Down to Earth (2001),Fantasy,Sky Captain and the World of Tomorrow (2004),Action
